# N0: Error Analysis - Model Prediction Errors

Analyze where the best model makes mistakes on validation set:
1. Load validation set
2. Extract TF-IDF & PhoBERT embeddings
3. Load best model from model/machine_learned
4. Test on validation set
5. Extract and analyze incorrect predictions

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import warnings
import torch
import gc
import joblib
import os
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch (PhoBERT extraction)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Mode: Error Analysis on Validation Set")

✅ All imports successful
   PyTorch Device: cuda
   Mode: Error Analysis on Validation Set


## 2. Load Validation Set

In [2]:
train_path = '../../data/splited/train_set.csv'
val_path = '../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"✅ Train set loaded: {df_train.shape}")
print(f"   Labels: {(y_train==0).sum()} fake / {(y_train==1).sum()} real")

print(f"\n✅ Validation set loaded: {df_val.shape}")
print(f"   Labels: {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

✅ Train set loaded: (3788, 28)
   Labels: 3143 fake / 645 real

✅ Validation set loaded: (474, 28)
   Labels: 393 fake / 81 real


## 3. Generate TF-IDF Embeddings

In [3]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

# Fit TF-IDF on TRAINING data (same as training notebook)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

# Fit SVD on TRAINING data
svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings fitted on training data")
print(f"   Train: {X_train_tfidf.shape} | Val: {X_val_tfidf.shape}")

✅ TF-IDF embeddings fitted on training data
   Train: (3788, 900) | Val: (474, 900)


## 4. Extract PhoBERT Embeddings

In [4]:
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """Extract PhoBERT embeddings from text"""
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="PhoBERT", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    return np.array(embeddings)

X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"✅ PhoBERT embeddings: Val {X_val_phobert.shape}")

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PhoBERT embeddings: Val (474, 768)


## 5. Combine and Scale Embeddings

In [5]:
# Combine embeddings (TF-IDF + PhoBERT)
X_val_combined = np.hstack([X_val_tfidf, X_val_phobert])

print(f"✅ Combined embeddings: {X_val_combined.shape[1]} dimensions")
print(f"   TF-IDF: 900 + PhoBERT: 768 = Total: {900 + 768}")

✅ Combined embeddings: 1668 dimensions
   TF-IDF: 900 + PhoBERT: 768 = Total: 1668


## 6. Load Best Model & Scaler

In [13]:
print("\n" + "="*80)
print("LOADING BEST MODEL")
print("="*80)

# Find model files in model/machine_learned
model_dir = os.path.abspath('../../model/machine_learned')
print(f"\n📂 Model directory: {model_dir}")

# List files in directory
if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f"\n📁 Files found:")
    for f in sorted(files):
        print(f"   - {f}")
else:
    print(f"❌ Directory not found: {model_dir}")
    raise FileNotFoundError(f"Model directory not found: {model_dir}")

# Load the most recent model and scaler
model_files = [f for f in files if f.startswith('Model_') and f.endswith('.pkl')]
scaler_files = [f for f in files if f.startswith('scaler_') and f.endswith('.pkl')]

if not model_files:
    raise FileNotFoundError("No model files found in machine_learned directory")
if not scaler_files:
    raise FileNotFoundError("No scaler files found in machine_learned directory")

# Get the most recent model file (alphabetically, assuming YYYYMMDD_HHMMSS format)
latest_model_file = sorted(model_files)[-1]
latest_scaler_file = sorted(scaler_files)[-1]

model_path = os.path.join(model_dir, latest_model_file)
scaler_path = os.path.join(model_dir, latest_scaler_file)

print(f"\n🔄 Loading model: {latest_model_file}")
best_model = joblib.load(model_path)
print(f"✅ Model loaded successfully")

print(f"\n🔄 Loading scaler: {latest_scaler_file}")
scaler = joblib.load(scaler_path)
print(f"✅ Scaler loaded successfully")


LOADING BEST MODEL

📂 Model directory: d:\Vietnamese-Fake-News-Detection\model\machine_learned

📁 Files found:
   - Model_4_4_20260406_192428.pkl
   - Model_4_4_20260406_192428_metadata.txt
   - scaler_20260406_192428.pkl

🔄 Loading model: Model_4_4_20260406_192428.pkl
✅ Model loaded successfully

🔄 Loading scaler: scaler_20260406_192428.pkl
✅ Scaler loaded successfully


## 7. Scale and Predict on Validation Set

In [14]:
print("\n" + "="*80)
print("MAKING PREDICTIONS ON VALIDATION SET")
print("="*80)

# Scale validation data
X_val_combined_scaled = scaler.transform(X_val_combined)
print(f"\n✅ Data scaled: {X_val_combined_scaled.shape}")

# Make predictions
y_val_pred = best_model.predict(X_val_combined_scaled)
y_val_proba = best_model.predict_proba(X_val_combined_scaled)[:, 1]

print(f"✅ Predictions made for {len(y_val_pred)} samples")
print(f"\n📊 Prediction Distribution:")
print(f"   Fake (0): {(y_val_pred==0).sum()}")
print(f"   Real (1): {(y_val_pred==1).sum()}")


MAKING PREDICTIONS ON VALIDATION SET

✅ Data scaled: (474, 1668)
✅ Predictions made for 474 samples

📊 Prediction Distribution:
   Fake (0): 401
   Real (1): 73


## 8. Calculate Metrics

In [15]:
print("\n" + "="*80)
print("OVERALL PERFORMANCE METRICS")
print("="*80)

f1 = f1_score(y_val, y_val_pred, average='weighted')
auc = roc_auc_score(y_val, y_val_proba)
acc = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, average='weighted')
rec = recall_score(y_val, y_val_pred, average='weighted')

print(f"\n✅ Performance Metrics:")
print(f"   F1 Score:    {f1:.4f}")
print(f"   AUC-ROC:     {auc:.4f}")
print(f"   Accuracy:    {acc:.4f}")
print(f"   Precision:   {prec:.4f}")
print(f"   Recall:      {rec:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred)
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives (TN):  {cm[0,0]}  |  False Positives (FP): {cm[0,1]}")
print(f"   False Negatives (FN): {cm[1,0]}  |  True Positives (TP):  {cm[1,1]}")

# Classification Report
print(f"\n📋 Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=['Fake (0)', 'Real (1)']))


OVERALL PERFORMANCE METRICS

✅ Performance Metrics:
   F1 Score:    0.9138
   AUC-ROC:     0.9594
   Accuracy:    0.9156
   Precision:   0.9129
   Recall:      0.9156

📊 Confusion Matrix:
   True Negatives (TN):  377  |  False Positives (FP): 16
   False Negatives (FN): 24  |  True Positives (TP):  57

📋 Classification Report:
              precision    recall  f1-score   support

    Fake (0)       0.94      0.96      0.95       393
    Real (1)       0.78      0.70      0.74        81

    accuracy                           0.92       474
   macro avg       0.86      0.83      0.84       474
weighted avg       0.91      0.92      0.91       474



## 9. Identify Incorrect Predictions

In [16]:
print("\n" + "="*80)
print("ERROR ANALYSIS: INCORRECT PREDICTIONS")
print("="*80)

# Find incorrect predictions
incorrect_mask = y_val != y_val_pred
incorrect_indices = np.where(incorrect_mask)[0]

print(f"\n📊 Error Statistics:")
print(f"   Total samples: {len(y_val)}")
print(f"   Correct predictions: {(~incorrect_mask).sum()}")
print(f"   Incorrect predictions: {incorrect_mask.sum()}")
print(f"   Error rate: {incorrect_mask.sum() / len(y_val) * 100:.2f}%")

# Analyze error types
false_positives_mask = (y_val == 0) & (y_val_pred == 1)  # Real labeled as Fake
false_negatives_mask = (y_val == 1) & (y_val_pred == 0)  # Fake labeled as Real

print(f"\n❌ Error Types:")
print(f"   False Positives (Real → Fake): {false_positives_mask.sum()}")
print(f"   False Negatives (Fake → Real): {false_negatives_mask.sum()}")


ERROR ANALYSIS: INCORRECT PREDICTIONS

📊 Error Statistics:
   Total samples: 474
   Correct predictions: 434
   Incorrect predictions: 40
   Error rate: 8.44%

❌ Error Types:
   False Positives (Real → Fake): 16
   False Negatives (Fake → Real): 24


## 10. Analyze False Positives (Real labeled as Fake)

In [20]:
print("\n" + "="*80)
print("FALSE POSITIVES: Real News Predicted as Fake")
print("="*80)

fp_indices = np.where(false_positives_mask)[0]
print(f"\n📊 Total False Positives: {len(fp_indices)}")

if len(fp_indices) > 0:
    print(f"\n📝 All False Positives ({len(fp_indices)} examples):")
    print("="*80)
    
    for idx, i in enumerate(fp_indices, 1):
        print(f"\n▶ Example {idx}:")
        print(f"   ID: {df_val.iloc[i]['id']}")
        print(f"   Actual Label: Real (1)")
        print(f"   Predicted: Fake (0)")
        print(f"   Confidence: {1 - y_val_proba[i]:.4f}")
        print(f"   Text: {df_val.iloc[i]['post_message'][:200]}...")
        
else:
    print(f"\n✅ No False Positives! All real news correctly classified.")


FALSE POSITIVES: Real News Predicted as Fake

📊 Total False Positives: 16

📝 All False Positives (16 examples):

▶ Example 1:
   ID: 209
   Actual Label: Real (1)
   Predicted: Fake (0)
   Confidence: 0.0311
   Text: Dân Ấn_Độ phẫn_nộ đốt hình Tập_Cận_Bình và cờ Trung_Quốc , đồng_thời kêu_gọi tẩy_chay hàng Trung_quốc sau khi quân TQ giết 20 người Ấn_Độ trong cuộc đụng_độ . Ấn_Độ đông dân thứ 2 thế_giới đó , Tàu hã...

▶ Example 2:
   ID: 1943
   Actual Label: Real (1)
   Predicted: Fake (0)
   Confidence: 0.1220
   Text: CHUYỆN NHỮNG NGƯỜI LAO_ĐỘNG VN Ở CHÂU_ÂU HỒI_HƯƠNG VÌ_CHINESE_VIRUS ( Các bạn bảo nhau ở nhà không đi làm cũng là cách_ly rồi , về VN cũng là cách_ly nhưng từ không có bệnh Chinese_Virus thành nhóm ng...

▶ Example 3:
   ID: 1176
   Actual Label: Real (1)
   Predicted: Fake (0)
   Confidence: 0.3045
   Text: Lần đầu_tiên kể từ năm 1990 , TQ không đưa ra mục_tiêu tăng_trưởng GDP , điều này dường_như phản_ánh mức nghiêm_trọng của các thiệt_hại từ Covid-19 mà Trung_Quốc 

## 11. Analyze False Negatives (Fake labeled as Real)

In [21]:
print("\n" + "="*80)
print("FALSE NEGATIVES: Fake News Predicted as Real")
print("="*80)

fn_indices = np.where(false_negatives_mask)[0]
print(f"\n📊 Total False Negatives: {len(fn_indices)}")

if len(fn_indices) > 0:
    print(f"\n📝 All False Negatives ({len(fn_indices)} examples):")
    print("="*80)
    
    for idx, i in enumerate(fn_indices, 1):
        print(f"\n▶ Example {idx}:")
        print(f"   ID: {df_val.iloc[i]['id']}")
        print(f"   Actual Label: Fake (0)")
        print(f"   Predicted: Real (1)")
        print(f"   Confidence: {y_val_proba[i]:.4f}")
        print(f"   Text: {df_val.iloc[i]['post_message'][:200]}...")
        
else:
    print(f"\n✅ No False Negatives! All fake news correctly classified.")


FALSE NEGATIVES: Fake News Predicted as Real

📊 Total False Negatives: 24

📝 All False Negatives (24 examples):

▶ Example 1:
   ID: 3555
   Actual Label: Fake (0)
   Predicted: Real (1)
   Confidence: 0.3824
   Text: " Có bằng_chứng to_lớn cho thấy đó là nơi nó ( virus ) xuất_phát . Tôi nghĩ cả thế_giới giờ có_thể thấy rõ , Trung_Quốc có lịch_sử lây bệnh cho thế_giới , và họ vận_hành những phòng_thí_nghiệm không đ...

▶ Example 2:
   ID: 1363
   Actual Label: Fake (0)
   Predicted: Real (1)
   Confidence: 0.1681
   Text: SỐNG_CHẾT MẶC BAY Đó là bệnh_nhân_số 31 của Hàn_Quốc , khi trực_tiếp và gián_tiếp lây_nhiễm cho hàng trăm người và sẽ nhiều hơn thế nữa . Và bệnh_nhân_số 31 này cũng trở_thành một trong những trường_h...

▶ Example 3:
   ID: 3804
   Actual Label: Fake (0)
   Predicted: Real (1)
   Confidence: 0.1081
   Text: Chính_phủ cần giải_thích tại_sao chưa có kết_luận thanh_tra giá điện tăng bất_minh năm 2019 , giờ_đây lại cho_phép EVN tiếp_tục tăng_giá điện ?...

▶ Example 4:


## 12. Summary of Error Patterns

In [22]:
print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 Overall Performance:")
print(f"   ✅ Correct: {(~incorrect_mask).sum()} / {len(y_val)} ({(~incorrect_mask).sum() / len(y_val) * 100:.2f}%)")
print(f"   ❌ Incorrect: {incorrect_mask.sum()} / {len(y_val)} ({incorrect_mask.sum() / len(y_val) * 100:.2f}%)")

print(f"\n📈 Error Breakdown:")
print(f"   False Positives (Real → Fake): {false_positives_mask.sum()} ({false_positives_mask.sum() / len(y_val) * 100:.2f}%)")
print(f"   False Negatives (Fake → Real): {false_negatives_mask.sum()} ({false_negatives_mask.sum() / len(y_val) * 100:.2f}%)")

print(f"\n🎯 Key Insights:")
if false_positives_mask.sum() > false_negatives_mask.sum():
    print(f"   → Model tends to over-predict FAKE news")
    print(f"   → More conservative (skeptical) predictions")
elif false_negatives_mask.sum() > false_positives_mask.sum():
    print(f"   → Model tends to under-predict FAKE news")
    print(f"   → Model is too trusting (optimistic)")
else:
    print(f"   → Model is balanced in both directions")

print(f"\n💡 Recommendations:")
if false_negatives_mask.sum() > false_positives_mask.sum():
    print(f"   1. Tune decision threshold to be more conservative")
    print(f"   2. Add more training examples of fake news")
    print(f"   3. Use ensemble methods to catch more fake news")
else:
    print(f"   1. Reduce false alarms by adjusting threshold")
    print(f"   2. Improve robustness of real news detection")
    print(f"   3. Check for potential bias in model")

print(f"\n✅ Error Analysis Complete!")


ERROR ANALYSIS SUMMARY

📊 Overall Performance:
   ✅ Correct: 434 / 474 (91.56%)
   ❌ Incorrect: 40 / 474 (8.44%)

📈 Error Breakdown:
   False Positives (Real → Fake): 16 (3.38%)
   False Negatives (Fake → Real): 24 (5.06%)

🎯 Key Insights:
   → Model tends to under-predict FAKE news
   → Model is too trusting (optimistic)

💡 Recommendations:
   1. Tune decision threshold to be more conservative
   2. Add more training examples of fake news
   3. Use ensemble methods to catch more fake news

✅ Error Analysis Complete!


## 13. Threshold Tuning Analysis

In [26]:
print("\n" + "="*80)
print("THRESHOLD TUNING ANALYSIS")
print("="*80)

# Test different thresholds
thresholds = np.arange(0.05, 1.00, 0.05)
results = []

print("\n📊 Testing different thresholds...")
print("="*110)

for threshold in thresholds:
    # Apply threshold
    y_val_pred_tuned = (y_val_proba >= threshold).astype(int)
    
    # Calculate metrics
    f1 = f1_score(y_val, y_val_pred_tuned, average='weighted')
    auc = roc_auc_score(y_val, y_val_proba)  # AUC doesn't change with threshold
    acc = accuracy_score(y_val, y_val_pred_tuned)
    prec = precision_score(y_val, y_val_pred_tuned, average='weighted', zero_division=0)
    rec = recall_score(y_val, y_val_pred_tuned, average='weighted', zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_val, y_val_pred_tuned)
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    total_errors = fp + fn
    
    results.append({
        'Threshold': threshold,
        'F1': f1,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'FP': fp,
        'FN': fn,
        'FP+FN': total_errors,
        'TP': tp,
        'TN': tn
    })
    
    print(f"\n▶ Threshold: {threshold:.2f}")
    print(f"   F1:        {f1:.4f}")
    print(f"   Accuracy:  {acc:.4f}")
    print(f"   Precision: {prec:.4f}")
    print(f"   Recall:    {rec:.4f}")
    print(f"   Confusion: FP={fp:3d}  |  FN={fn:3d}  |  Total Errors={total_errors:3d}")

# Create comparison dataframe
df_results = pd.DataFrame(results)

print("\n\n" + "="*130)
print("COMPARISON TABLE")
print("="*130)

# Create display version with marker
df_display = df_results[['Threshold', 'F1', 'Accuracy', 'Precision', 'Recall', 'FP', 'FN', 'FP+FN']].copy()
df_display['Status'] = ''

# Mark the 0.50 threshold row
threshold_050_idx = (df_display['Threshold'] - 0.50).abs().idxmin()
df_display.loc[threshold_050_idx, 'Status'] = '← CURRENT (0.50)'

print(df_display.to_string(index=False))

# Find best threshold based on F1 score
best_idx = df_results['F1'].idxmax()
best_threshold = df_results.loc[best_idx, 'Threshold']
best_f1 = df_results.loc[best_idx, 'F1']
best_total_errors = df_results.loc[best_idx, 'FP+FN']

# Current threshold (0.50) metrics
current_idx = (df_results['Threshold'] - 0.50).abs().idxmin()
current_f1 = df_results.loc[current_idx, 'F1']
current_fp = df_results.loc[current_idx, 'FP']
current_fn = df_results.loc[current_idx, 'FN']
current_total_errors = df_results.loc[current_idx, 'FP+FN']

print(f"\n\n🎯 OPTIMAL THRESHOLD: {best_threshold:.2f}")
print(f"   Best F1 Score: {best_f1:.4f}")
print(f"   FP (False Positives): {int(df_results.loc[best_idx, 'FP'])}")
print(f"   FN (False Negatives): {int(df_results.loc[best_idx, 'FN'])}")
print(f"   Total Errors (FP+FN): {int(best_total_errors)}")

print("\n\n💡 COMPARISON: Current (0.50) vs Optimal ({:.2f})".format(best_threshold))
print("="*80)
print(f"{'Metric':<25} {'Current (0.50)':<20} {'Optimal ({:.2f})'.format(best_threshold):<20}")
print("="*80)
print(f"{'F1 Score':<25} {current_f1:<20.4f} {best_f1:<20.4f}")
print(f"{'False Positives (FP)':<25} {current_fp:<20} {int(df_results.loc[best_idx, 'FP']):<20}")
print(f"{'False Negatives (FN)':<25} {current_fn:<20} {int(df_results.loc[best_idx, 'FN']):<20}")
print(f"{'Total Errors (FP+FN)':<25} {current_total_errors:<20} {int(best_total_errors):<20}")
print("="*80)

improvement_fn = current_fn - int(df_results.loc[best_idx, 'FN'])
change_fp = int(df_results.loc[best_idx, 'FP']) - current_fp
improvement_total = current_total_errors - int(best_total_errors)

print(f"\n📊 IMPROVEMENT ANALYSIS:")
print(f"   ✅ False Negatives reduced by: {improvement_fn} cases ({improvement_fn/current_fn*100:.1f}%)")
print(f"   ⚠️  False Positives increased by: {change_fp} cases (+{change_fp/current_fp*100:.1f}%)")
print(f"   📈 Total Errors reduced by: {improvement_total} cases ({improvement_total/current_total_errors*100:.1f}%)")

print("\n💡 RECOMMENDATION:")
print(f"   • Changing threshold from 0.50 → {best_threshold:.2f} is HIGHLY RECOMMENDED")
print(f"   • Benefit: Catch {improvement_fn} more fake news cases (reduce false negatives)")
print(f"   • Cost: {change_fp} more false alarms (acceptable trade-off)")
print(f"   • Net benefit: {improvement_total} fewer total errors!")

print("\n✅ Threshold Tuning Analysis Complete!")


THRESHOLD TUNING ANALYSIS

📊 Testing different thresholds...

▶ Threshold: 0.05
   F1:        0.8418
   Accuracy:  0.8228
   Precision: 0.9050
   Recall:    0.8228
   Confusion: FP= 81  |  FN=  3  |  Total Errors= 84

▶ Threshold: 0.10
   F1:        0.8967
   Accuracy:  0.8882
   Precision: 0.9220
   Recall:    0.8882
   Confusion: FP= 48  |  FN=  5  |  Total Errors= 53

▶ Threshold: 0.15
   F1:        0.9046
   Accuracy:  0.8987
   Precision: 0.9194
   Recall:    0.8987
   Confusion: FP= 39  |  FN=  9  |  Total Errors= 48

▶ Threshold: 0.20
   F1:        0.8997
   Accuracy:  0.8945
   Precision: 0.9106
   Recall:    0.8945
   Confusion: FP= 37  |  FN= 13  |  Total Errors= 50

▶ Threshold: 0.25
   F1:        0.9067
   Accuracy:  0.9030
   Precision: 0.9138
   Recall:    0.9030
   Confusion: FP= 32  |  FN= 14  |  Total Errors= 46

▶ Threshold: 0.30
   F1:        0.9156
   Accuracy:  0.9135
   Precision: 0.9191
   Recall:    0.9135
   Confusion: FP= 26  |  FN= 15  |  Total Errors= 41

▶